# ULEZ — Urban Mobility and Emissions Simulation

An exploratory model on a 20 × 20 abstract urban grid, using the fleet assumptions and emission coefficient tables defined below. This notebook compares 27 alternative scenarios (three zone sizes × nine restriction rules).

Run from the repository directory after installing `requirements.txt`. The default is 5,000 synthetic trips and a fixed random seed. Outputs go to `outputs/`. No external API is used.

Emission values are **model units**. The inherited coefficient formula and proxy assumptions for motorcycles/commercial vehicles have not been independently calibrated to a specific city or validated as physical NOx measurements. Scenario accounting fixes change the old exploratory totals; previously saved figures should not be treated as current results.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import os

SEED = int(os.environ.get("ULEZ_SEED", "42"))
N_TRIPS = int(os.environ.get("ULEZ_N_TRIPS", "5000"))
MAX_WAIT_HOURS = 2
if N_TRIPS <= 0:
    raise ValueError("ULEZ_N_TRIPS must be positive")
random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR = Path(os.environ.get("ULEZ_OUTPUT_DIR", str(Path.cwd() / "outputs")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_figure_number = 0

def show_figure():
    global _figure_number
    _figure_number += 1
    plt.gcf().savefig(OUTPUT_DIR / f"figure_{_figure_number:02d}.png", dpi=150, bbox_inches="tight")
    if plt.get_backend().lower() != "agg":
        plt.show()
    plt.close("all")


# **Basic data**

In [ ]:
# Define the data for traffic distribution across different times of the day
data_td = {
    'time': ['00:00-00:59', '01:00-01:59', '02:00-02:59', '03:00-03:59', '04:00-04:59', '05:00-05:59',
             '06:00-06:59', '07:00-07:59', '08:00-08:59', '09:00-09:59', '10:00-10:59', '11:00-11:59',
             '12:00-12:59', '13:00-13:59', '14:00-14:59', '15:00-15:59', '16:00-16:59', '17:00-17:59',
             '18:00-18:59', '19:00-19:59', '20:00-20:59', '21:00-21:59', '22:00-22:59', '23:00-23:59'],
    'probability': [0.8, 0.4, 0.3, 0.4, 0.8, 2.0, 4.5, 7.0, 7.0, 6.0, 5.7, 5.9, 6.0, 6.1, 6.5, 7.0,
                    7.7, 7.6, 6.0, 4.2, 3.0, 2.2, 1.8, 1.1]
}

# Create a DataFrame
traffic_distribution_df = pd.DataFrame(data_td)
traffic_distribution_df['hour'] = range(24)  # Add hour column

# Normalize the probabilities to sum to 1
traffic_distribution_df['probability'] /= traffic_distribution_df['probability'].sum()

# Maximum and minimum speeds in the city (in km/h)
max_speed = 50
min_speed = 10

# Calculate average speed based on traffic probability
traffic_distribution_df['average_speed1'] = max_speed * (1 - traffic_distribution_df['probability'] / max(traffic_distribution_df['probability']))
traffic_distribution_df['average_speed1'] = traffic_distribution_df['average_speed1'].apply(lambda x: max(min_speed, min(max_speed, x)))

# Calculate average speed for the next hour
traffic_distribution_df['average_speed2'] = traffic_distribution_df['average_speed1'].shift(-1)

# For the last hour, use the speed of the first hour (or another logical value)
traffic_distribution_df.at[23, 'average_speed2'] = traffic_distribution_df.at[0, 'average_speed1']

# Round speeds to two decimal places
traffic_distribution_df['average_speed1'] = traffic_distribution_df['average_speed1'].round(2)
traffic_distribution_df['average_speed2'] = traffic_distribution_df['average_speed2'].round(2)

# Ensure 'average_speed' columns exist and have no zero values
traffic_distribution_df['average_speed1'] = traffic_distribution_df['average_speed1'].replace(0, 1e-3)
traffic_distribution_df['average_speed2'] = traffic_distribution_df['average_speed2'].replace(0, 1e-3)

# Specify the path to save the CSV file
csv_path = OUTPUT_DIR / 'traffic_distribution.csv'

# Save the DataFrame to a CSV file
traffic_distribution_df.to_csv(csv_path, index=False)

In [ ]:
# Define the data for vehicle categories and their initial percentages
data_vd = {
    'category': ['Petrol passenger cars', 'Diesel passenger cars', 'Commercial Vehicles (diesel)', 'Motorcycles (petrol)'],
    'probability': [45, 39, 12, 2]
}

# Create a DataFrame
vehicle_categories_df = pd.DataFrame(data_vd)

# Normalize the probabilities to sum to 1
vehicle_categories_df['probability'] /= vehicle_categories_df['probability'].sum()

# Specify the path to save the CSV file
csv_path = OUTPUT_DIR / 'vehicle_categories.csv'

# Save the DataFrame to a CSV file
vehicle_categories_df.to_csv(csv_path, index=False)

In [ ]:
# Data for vehicle ages including motorcycles
data_ed = {
    'Vehicle Type': ['Passenger car Slovakia', 'CV Slovakia', 'Motorcycle Slovakia'],
    '2021-2018': [338692, 44079, 0],   # Euro 6
    '2017-2016': [210013, 25051, 0],     # Euro 5
    '2015-2013': [274634, 30208, 0],     # Euro 4
    '2012-2010': [287296, 30836, 5652],  # Euro 3 for motorcycles
    '<2010': [1727293, 258533, 62074]    # Euro 2 and Euro 1 for motorcycles
}

vehicle_age_df = pd.DataFrame(data_ed)

# Calculate the total for normalization
vehicle_age_df['Total'] = vehicle_age_df.drop('Vehicle Type', axis=1).sum(axis=1)

# Assign Euro class probabilities based on vehicle age groups
vehicle_age_df['Euro 6'] = vehicle_age_df['2021-2018'] / vehicle_age_df['Total']
vehicle_age_df['Euro 5'] = vehicle_age_df['2017-2016'] / vehicle_age_df['Total']
vehicle_age_df['Euro 4'] = vehicle_age_df['2015-2013'] / vehicle_age_df['Total']
vehicle_age_df['Euro 3'] = vehicle_age_df['2012-2010'] / vehicle_age_df['Total']
vehicle_age_df['Euro 2'] = vehicle_age_df['<2010'] * 0.65 / vehicle_age_df['Total']
vehicle_age_df['Euro 1'] = vehicle_age_df['<2010'] * 0.35 / vehicle_age_df['Total']

# Adjust for motorcycles only having Euro 1 and Euro 3 classifications
vehicle_age_df.loc[vehicle_age_df['Vehicle Type'] == 'Motorcycle Slovakia', ['Euro 6', 'Euro 5', 'Euro 4', 'Euro 2']] = 0
vehicle_age_df.loc[vehicle_age_df['Vehicle Type'] == 'Motorcycle Slovakia', 'Euro 3'] = 0.52  # 52% of motorcycles are Euro 3
vehicle_age_df.loc[vehicle_age_df['Vehicle Type'] == 'Motorcycle Slovakia', 'Euro 1'] = 0.48  # 48% of motorcycles are Euro 1

# Dropping the individual year columns for clarity
euro_classes_df = vehicle_age_df[['Vehicle Type', 'Euro 6', 'Euro 5', 'Euro 4', 'Euro 3', 'Euro 2', 'Euro 1']]

# Setting index for better plotting
euro_classes_df = euro_classes_df.set_index('Vehicle Type')

# Specify the path to save the CSV file
csv_path = OUTPUT_DIR / 'euro_classes.csv'

# Save the DataFrame to a CSV file
euro_classes_df.to_csv(csv_path, index=True)

# **Algorithm 1**

In [ ]:
# Define function to generate random coordinates within the 20x20 space as integers
def generate_random_coordinates():
    return (random.randint(0, 20), random.randint(0, 20)), (random.randint(0, 20), random.randint(0, 20))

# Function to calculate Euclidean distance
def euclidean_distance(origin, destination):
    return math.sqrt((origin[0] - destination[0])**2 + (origin[1] - destination[1])**2)

# Function to generate a single trip
def generate_trip():
    origin, destination = generate_random_coordinates()
    distance_km = euclidean_distance(origin, destination)
    return {
        'origin': origin,
        'destination': destination,
        'distance_km': round(distance_km, 3)  # Round to 3 decimal places
    }

# Generate the configured number of trips
trips = [generate_trip() for _ in range(N_TRIPS)]

# Convert to DataFrame
trips_df = pd.DataFrame(trips)

# Function to determine if two line segments intersect
def lines_intersect(p1, q1, p2, q2):
    def orientation(p, q, r):
        val = (q[1] - p[1]) * (r[0] - q[0]) - (q[0] - p[0]) * (r[1] - q[1])
        if val == 0:
            return 0
        return 1 if val > 0 else 2

    def on_segment(p, q, r):
        if min(p[0], q[0]) <= r[0] <= max(p[0], q[0]) and min(p[1], q[1]) <= r[1] <= max(p[1], q[1]):
            return True
        return False

    o1 = orientation(p1, q1, p2)
    o2 = orientation(p1, q1, q2)
    o3 = orientation(p2, q2, p1)
    o4 = orientation(p2, q2, q1)

    if o1 != o2 and o3 != o4:
        return True

    if o1 == 0 and on_segment(p1, q1, p2):
        return True
    if o2 == 0 and on_segment(p1, q1, q2):
        return True
    if o3 == 0 and on_segment(p2, q2, p1):
        return True
    if o4 == 0 and on_segment(p2, q2, q1):
        return True

    return False

# Function to determine the case for each trip based on LEZ size with improved logic
def determine_case(trip, lez_size):
    center = 10  # Center for 20x20 space
    half_size = lez_size / 2

    origin = trip['origin']
    destination = trip['destination']

    origin_inside = (center - half_size <= origin[0] <= center + half_size and
                     center - half_size <= origin[1] <= center + half_size)
    destination_inside = (center - half_size <= destination[0] <= center + half_size and
                          center - half_size <= destination[1] <= center + half_size)

    if origin_inside and destination_inside:
        return 1  # Both points inside the LEZ
    elif origin_inside and not destination_inside:
        return 2  # Departure from inside LEZ, arrival outside LEZ
    elif not origin_inside and destination_inside:
        return 3  # Departure from outside LEZ, arrival inside LEZ
    elif not origin_inside and not destination_inside:
        # Check if the route passes through the LEZ
        zone_edges = [
            ((center - half_size, center - half_size), (center + half_size, center - half_size)),
            ((center + half_size, center - half_size), (center + half_size, center + half_size)),
            ((center + half_size, center + half_size), (center - half_size, center + half_size)),
            ((center - half_size, center + half_size), (center - half_size, center - half_size))
        ]

        for edge in zone_edges:
            if lines_intersect(origin, destination, edge[0], edge[1]):
                return 4  # Route outside LEZ, passing through LEZ

        return 5  # Both points and route entirely outside LEZ

# Correct the sizes of the zones based on the area percentages
lez_sizes = {name: 20 * math.sqrt(share) for name, share in [('Area 1', 0.15), ('Area 2', 0.30), ('Area 3', 0.40)]}  # Scaled for 20x20 space
lez_sizes_display = {'Area 1': 15, 'Area 2': 30, 'Area 3': 40}

# Apply LEZ categories to the trips and add columns to the existing DataFrame
for area_name, lez_size in lez_sizes.items():
    trips_df[area_name + '_case'] = trips_df.apply(lambda x: determine_case(x, lez_size), axis=1)

# Function to visualize trips with different cases
def visualize_trips_by_case(trips_df, lez_sizes, lez_sizes_display, case_num):
    colors = {1: 'green', 2: 'blue', 3: 'orange', 4: 'red', 5: 'purple'}

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, (area_name, lez_size) in enumerate(lez_sizes.items()):
        ax = axes[i]

        # Define the center of the space
        center = 10  # Center for 20x20 space
        half_size = lez_size / 2

        # Add a transparent green square representing the LEZ area on top
        rect = patches.Rectangle(
            (center - half_size, center - half_size),
            lez_size,
            lez_size,
            linewidth=2,
            edgecolor='none',
            facecolor='green',
            alpha=0.3,
            label=f'{area_name} ({lez_sizes_display[area_name]}%)'
        )
        ax.add_patch(rect)

        # Plot the trips with the specified case number
        for _, trip in trips_df[trips_df[area_name + '_case'] == case_num].iterrows():
            origin = trip['origin']
            destination = trip['destination']
            ax.plot([origin[0], destination[0]], [origin[1], destination[1]], color=colors[case_num], alpha=0.5)
            ax.arrow(origin[0], origin[1], destination[0] - origin[0], destination[1] - origin[1],
                     head_width=0.5, head_length=0.5, fc=colors[case_num], ec=colors[case_num])

        # Set limits and labels
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 20)
        ax.set_xlabel('X Coordinate')
        ax.set_ylabel('Y Coordinate')
        ax.set_title(f'Visualization of {area_name} - Case {case_num}')
        ax.legend()
        ax.grid(True)
        ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    show_figure()

# Function to calculate and display case distribution with debug prints
def calculate_and_display_case_distribution(trips_df, lez_sizes):
    case_counts = {key: {i: 0 for i in range(1, 6)} for key in lez_sizes.keys()}

    for _, trip in trips_df.iterrows():
        for area_name in lez_sizes.keys():
            case = trip[area_name + '_case']
            case_counts[area_name][case] += 1

    for area_name, counts in case_counts.items():
        print(f"\nDistribution of cases for {area_name}:")
        total = sum(counts.values())
        for case, count in counts.items():
            proportion = count / total if total > 0 else 0
            print(f"  Case {case}: {count} trips ({proportion:.2%})")

# Visualize the trips by case number
for case_num in range(1, 6):
    print(f'Visualizing Case {case_num}')
    visualize_trips_by_case(trips_df, lez_sizes, lez_sizes_display, case_num)

# Calculate and display case distribution
calculate_and_display_case_distribution(trips_df, lez_sizes)

# Save the DataFrame to a CSV file
trips_df.to_csv(OUTPUT_DIR / 'trips_data_with_lez_cases.csv', index=False)

In [ ]:
# Function to calculate proportions and fill missing cases
def calculate_proportions(df, case_column):
    case_counts = df[case_column].value_counts(normalize=True).sort_index()
    # Fill missing cases with zeros
    for i in range(1, 6):
        if i not in case_counts:
            case_counts[i] = 0
    return case_counts.sort_index()

# Define columns for each area
case_columns = ['Area 1_case', 'Area 2_case', 'Area 3_case']

# Calculate proportions for each area
props_1 = calculate_proportions(trips_df, case_columns[0])
props_2 = calculate_proportions(trips_df, case_columns[1])
props_3 = calculate_proportions(trips_df, case_columns[2])

# Print distribution of cases
print("Distribution of cases for:")
print(props_1)
print("Distribution of cases for:")
print(props_2)
print("Distribution of cases for:")
print(props_3)

# Create the plot
fig, ax = plt.subplots()
index = np.arange(1, 6)  # Case numbers
bar_width = 0.25

rects1 = ax.bar(index, props_1, bar_width, label='Area I')
rects2 = ax.bar(index + bar_width, props_2, bar_width, label='Area II')
rects3 = ax.bar(index + 2 * bar_width, props_3, bar_width, label='Area III')

ax.set_xlabel('Case')
ax.set_ylabel('Proportion of OD cases')
ax.set_title('OD pairs cases per area size')
ax.set_xticks(index + bar_width)
ax.set_xticklabels(['Case 1', 'Case 2', 'Case 3', 'Case 4', 'Case 5'])
ax.legend()

# Display the plot
show_figure()

In [ ]:
# Function to randomly assign time of day, vehicle category, and Euro class to each trip
def assign_random_values(trip):
    trip['time'] = np.random.choice(traffic_distribution_df['time'], p=traffic_distribution_df['probability'])
    trip['vehicle_category'] = np.random.choice(vehicle_categories_df['category'], p=vehicle_categories_df['probability'])
    vehicle_type = 'Passenger car Slovakia' if 'passenger' in trip['vehicle_category'] else 'CV Slovakia' if 'Commercial' in trip['vehicle_category'] else 'Motorcycle Slovakia'
    euro_probabilities = euro_classes_df.loc[vehicle_type]
    trip['euro_class'] = np.random.choice(euro_probabilities.index, p=euro_probabilities.values)
    return trip

# Apply the function to the DataFrame with trips
trips_df = trips_df.apply(assign_random_values, axis=1)

# Save the DataFrame to a CSV file
trips_df.to_csv(OUTPUT_DIR / 'trips_data_with_random_values.csv', index=False)

In [ ]:
trips_df.head(2)

# **Adding variables**

In [ ]:
# DataFrame trips_df 0 1
trips_df['time2'] = trips_df['time'].str.split('-').str[0].str.split(':').str[0].astype(int)
trips_df['A_Case_1_5'] = (trips_df['Area 1_case'] == 5).astype(int)
trips_df['A_Case_2_5'] = (trips_df['Area 2_case'] == 5).astype(int)
trips_df['A_Case_3_5'] = (trips_df['Area 3_case'] == 5).astype(int)
trips_df['Diesel'] = trips_df['vehicle_category'].isin(['Diesel passenger cars', 'Commercial Vehicles (diesel)']).astype(int)
trips_df['Lux'] = ((trips_df['vehicle_category'].isin(['Petrol passenger cars']) & (trips_df['euro_class'] == 'Euro 6')) |
                     (trips_df['vehicle_category'] == 'Motorcycles (petrol)') & (trips_df['euro_class'] == 'Euro 3')).astype(int)

# Let's add the time period check columns 'period1' and 'period2' to the DataFrames
def add_time_period_checks(trips_df):
    # Adding 'period1' for the time period from 7AM to 7PM
    trips_df['period1'] = ((trips_df['time2'] >= 7) & (trips_df['time2'] <= 19)).astype(int)

    # Adding 'period2' for peak hours from 7AM to 10AM and from 4PM to 7PM
    trips_df['period2'] = (((trips_df['time2'] >= 7) & (trips_df['time2'] <= 10)) | ((trips_df['time2'] >= 16) & (trips_df['time2'] <= 19))).astype(int)

# Apply the function to each DataFrame
add_time_period_checks(trips_df)

In [ ]:
# Applying the restriction rules to each DataFrame
for df in [trips_df]:
    trips_df['1_RR1'] = (trips_df['A_Case_1_5'] == 0).astype(int)
    trips_df['1_RR2'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Lux'] == 0)).astype(int)
    trips_df['1_RR3'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Diesel'] == 1)).astype(int)
    trips_df['1_RR4'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['1_RR5'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['1_RR6'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period1'] == 1)).astype(int)
    trips_df['1_RR7'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['1_RR8'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['1_RR9'] = ((trips_df['A_Case_1_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period2'] == 1)).astype(int)

    trips_df['2_RR1'] = (trips_df['A_Case_2_5'] == 0).astype(int)
    trips_df['2_RR2'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Lux'] == 0)).astype(int)
    trips_df['2_RR3'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Diesel'] == 1)).astype(int)
    trips_df['2_RR4'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['2_RR5'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['2_RR6'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period1'] == 1)).astype(int)
    trips_df['2_RR7'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['2_RR8'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['2_RR9'] = ((trips_df['A_Case_2_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period2'] == 1)).astype(int)

    trips_df['3_RR1'] = (trips_df['A_Case_3_5'] == 0).astype(int)
    trips_df['3_RR2'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Lux'] == 0)).astype(int)
    trips_df['3_RR3'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Diesel'] == 1)).astype(int)
    trips_df['3_RR4'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['3_RR5'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period1'] == 1)).astype(int)
    trips_df['3_RR6'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period1'] == 1)).astype(int)
    trips_df['3_RR7'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['3_RR8'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Lux'] == 0) & (trips_df['period2'] == 1)).astype(int)
    trips_df['3_RR9'] = ((trips_df['A_Case_3_5'] == 0) & (trips_df['Diesel'] == 1) & (trips_df['period2'] == 1)).astype(int)

## Baseline travel and emissions

Each trip is integrated over consecutive hourly speeds, including midnight crossings. The entire trip distance is preserved, without rounding travel time before computing emissions. The original emission-factor formula below is retained as an exploratory assumption.


In [ ]:
# NOx emission factors for petrol cars and minibuses <2.5 tonnes GVW with 1400-2000 engine capacity.
petrol_cars_emission_factors = {
    'Euro 1': {'K': 1.00, 'a': 2.37, 'b': 0.199, 'c': 0.000647, 'd': 3.22e-06, 'e': 0, 'f': 0, 'g': 0},
    'Euro 2': {'K': 1.00, 'a': 1.10, 'b': 0.120, 'c': 0.000613, 'd': 1.17e-06, 'e': 8.78e-09, 'f': 0, 'g': 0},
    'Euro 3': {'K': 1.00, 'a': 0.437, 'b': 0.0614, 'c': 8.02e-05, 'd': 8.83e-08, 'e': 0, 'f': 0, 'g': 0},
    'Euro 4': {'K': 1.00, 'a': 0.517, 'b': 0.0345, 'c': 5.49e-05, 'd': 4.08e-07, 'e': 0, 'f': 0, 'g': 0},
    'Euro 5': {'K': 0.594, 'a': 0.517, 'b': 0.0345, 'c': 5.49e-05, 'd': 4.08e-07, 'e': 0, 'f': 0, 'g': 0},
    'Euro 6': {'K': 0.594, 'a': 0.517, 'b': 0.0345, 'c': 5.49e-05, 'd': 4.08e-07, 'e': 0, 'f': 0, 'g': 0}
}

# NOx emission factors for diesel cars and minibuses <2.5 tonnes GVW with 1400-2000 engine capacity.
diesel_cars_emission_factors = {
    'Euro 1': {'K': 1.00, 'a': 11.2, 'b': 0.158, 'c': 0.00124, 'd': 2.27e-05, 'e': -8.30e-08, 'f': 1.09e-09, 'g': 0},
    'Euro 2': {'K': 1.00, 'a': 14.8, 'b': 0.177, 'c': -0.0003, 'd': 6.43e-05, 'e': -4.62e-07, 'f': 2.96e-09, 'g': 0},
    'Euro 3': {'K': 1.00, 'a': 9.29, 'b': 0.168, 'c': 0.00118, 'd': 1.50e-05, 'e': -3.75e-08, 'f': 4.97e-10, 'g': 0},
    'Euro 4': {'K': 1.00, 'a': 5.49, 'b': 0.123, 'c': 0.000671, 'd': 2.08e-05, 'e': -9.97e-08, 'f': 8.50e-10, 'g': 0},
    'Euro 5': {'K': 0.675, 'a': 5.49, 'b': 0.123, 'c': 0.000671, 'd': 2.08e-05, 'e': -9.97e-08, 'f': 8.50e-10, 'g': 0},
    'Euro 6': {'K': 0.3, 'a': 5.49, 'b': 0.123, 'c': 0.000671, 'd': 2.08e-05, 'e': -9.97e-08, 'f': 8.50e-10, 'g': 0}
}

# Function to calculate emissions
def calculate_emissions(euro_class, speed, vehicle_category):
    if vehicle_category in ['Petrol passenger cars', 'Motorcycles (petrol)']:
        factors = petrol_cars_emission_factors.get(euro_class, None)
    elif vehicle_category in ['Diesel passenger cars', 'Commercial Vehicles (diesel)']:
        factors = diesel_cars_emission_factors.get(euro_class, None)
    else:
        return 0  # Default to 0 if vehicle category is not recognized

    if factors is None:
        return 0  # Default to 0 if euro_class is not found

    # Calculate emission using polynomial formula
    emissions = (factors['K'] * (factors['a'] + factors['b'] * speed**2 + factors['c'] * speed**3 +
                                 factors['d'] * speed**4 + factors['e'] * speed**5 +
                                 factors['f'] * speed**6)) / speed
    return emissions



In [ ]:
from simulation import evaluate_scenarios, travel_profile

trips_df['time2'] = trips_df['time'].str[:2].astype(int)
hourly_speeds = traffic_distribution_df.sort_values('hour')['average_speed1'].tolist()
summary_df, scenario_trips_df, baseline_values = evaluate_scenarios(
    trips_df, hourly_speeds, calculate_emissions, max_wait_hours=MAX_WAIT_HOURS
)
trips_df['total_emissions'] = baseline_values
trips_df.to_csv(OUTPUT_DIR / 'trips_baseline.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'scenario_summary.csv', index=False)
scenario_trips_df.to_csv(OUTPUT_DIR / 'scenario_trips.csv', index=False)
print(summary_df.to_string(index=False))


## Independent restriction scenarios

Category 1: proceed unchanged if the route avoids the zone or the rule does not apply. Category 2: cancel if the restriction cannot end within the two-hour waiting limit. Category 3: delay until the first unrestricted hour and recalculate the full trip at that departure time. Categories are exclusive and cover every trip.

The original hourly restriction windows are inclusive: 07–19 ends at 20:00; peak windows are 07–10 and 16–19. Decisions use the departure hour. The model does not estimate rerouting, congestion feedback, or a trip entering a restriction after departure. Alternative areas and rules are never added together as a single citywide total.


In [ ]:
for area, frame in summary_df.groupby('area', sort=True):
    frame = frame.set_index('rule')
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    frame[['proceed', 'cancel', 'delay']].div(N_TRIPS).plot.bar(stacked=True, ax=axes[0])
    axes[0].set_ylabel('Share of trips')
    axes[0].set_title(f'{area}: trip outcomes')
    frame[['baseline_emissions', 'scenario_emissions']].plot.bar(ax=axes[1])
    axes[1].set_ylabel('Emissions (model units)')
    axes[1].set_title(f'{area}: alternative restriction rules')
    plt.tight_layout()
    show_figure()


In [ ]:
# Accounting invariants: fail visibly if a future edit breaks scenario isolation.
assert len(trips_df) == N_TRIPS
assert len(summary_df) == 27
assert (summary_df[['proceed', 'cancel', 'delay']].sum(axis=1) == N_TRIPS).all()
assert (scenario_trips_df.loc[scenario_trips_df['status'].eq('cancel'), 'scenario_emissions'] == 0).all()
assert np.isfinite(scenario_trips_df['scenario_emissions']).all()
assert (scenario_trips_df['scenario_emissions'] >= 0).all()
assert np.allclose(summary_df['baseline_emissions'], sum(baseline_values))
assert np.allclose(trips_df['total_emissions'], baseline_values)
print(f'Validated {N_TRIPS:,} trips and 27 independent scenarios. Random seed: {SEED}. Outputs: {OUTPUT_DIR}')
